# Ordered Logistic Regression Results: FAIRˆ² Dataset Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIRˆ² dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library and its schema-driven access patterns.

### Dataset Source
The dataset is described by a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure mlcroissant is installed in this environment
!pip install --quiet mlcroissant

## 1. Data Loading

We'll load the dataset metadata and access the available record sets and fields via their `@id` attributes.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# The Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the metadata object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview

Let's examine which record sets (`cr:RecordSet` entities) are available in this dataset, and access their fields and columns by `@id`.

*Note:* The Croissant schema uses `@id` fields to uniquely identify record sets, fields, and columns in the dataset structure.

In [ ]:
# List available record sets with their @id and names.
record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets are defined in the metadata.")
else:
    print("Record sets detected:")
    for rs in record_sets:
        print(f"@id: {rs['@id']}, Name: {rs.get('name', '[unnamed]')}")
        if 'fields' in rs:
            print("  Fields:")
            for fld in rs['fields']:
                print(f"    @id: {fld['@id']}, Name: {fld.get('name', '[unnamed]')}, DataType: {fld.get('dataType', '[unknown]')}")

Below, we print out sample records for a selected record set using its `@id`. 

*Please update the variable `record_set_id` to the desired `@id` based on the list above.*

In [ ]:
# Example: Print records from a chosen record set (update this ID according to the displayed record sets):
record_set_id = None
if record_sets:
    # Select the first available record set @id for demonstration
    record_set_id = record_sets[0]['@id']

if record_set_id is not None:
    print(f"First 3 records from record set @id: {record_set_id}\n---")
    for i, rec in enumerate(dataset.records(record_set=record_set_id)):
        print(rec)
        if i >= 2:
            break
else:
    print("No record set @id found to display records.")


## 3. Data Extraction

Let's load all available record sets into Pandas DataFrames for analysis, referencing each by its `@id`.

In [ ]:
# Load all record sets into DataFrames, referenced by their @id
dataframes = {}

if record_sets:
    for rs in record_sets:
        rid = rs['@id']
        try:
            records = list(dataset.records(record_set=rid))
            df = pd.DataFrame(records)
            dataframes[rid] = df
            print(f"Loaded record set @id: {rid}, shape: {df.shape}")
        except Exception as e:
            print(f"Could not load records for @id: {rid} - {e}")
else:
    print("No record sets to load.")

# Display columns of the first available DataFrame
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"Columns in record set @id {first_rs_id}:")
    print(list(dataframes[first_rs_id].columns))
    display(dataframes[first_rs_id].head())
else:
    print("No dataframes were loaded.")


## 4. Exploratory Data Analysis (EDA)

We'll proceed to demonstrate filtering, normalization, and grouping operations using numerical and categorical fields referenced by their `@id`.

> *Refer to the previous overviews to pick valid field `@id` values for the selected record set.*

*If the dataset has no record sets loaded, this cell will do nothing.*

In [ ]:
# Choose a numeric field and a grouping field by their @id
if dataframes:
    # For demonstration: extract from the first record set
    df = dataframes[first_rs_id]
    print(f"DataFrame shape: {df.shape}")
    
    # Try to automatically detect a numeric field
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

    # Also pick a grouping field @id (preferably categorical/text)
    group_field_id = None
    for col in df.columns:
        if df[col].dtype == 'object' and col != numeric_field_id:
            group_field_id = col
            break

    if numeric_field_id:
        print(f"Using numeric field @id: {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"\nFiltered records where {numeric_field_id} > {threshold:.2f} (n={len(filtered_df)}):")
        display(filtered_df.head())
        
        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        if group_field_id:
            print(f"\nGrouping by field @id: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric fields found in DataFrame.")
else:
    print("No dataframes loaded for EDA.")


## 5. Visualization

Let's visualize the distribution of the chosen numeric field and, if grouping is available, compare means by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id:
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30, ax=axs[0])
    axs[0].set_title(f"Distribution of {numeric_field_id}")
    
    if group_field_id:
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df, ax=axs[1])
        axs[1].set_title(f"{numeric_field_id} by {group_field_id}")
        plt.setp(axs[1].xaxis.get_majorticklabels(), rotation=45)
    else:
        axs[1].axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No data available for visualization.')


## 6. Conclusion
In this notebook, we've demonstrated how to load, inspect, and preliminarily analyze a FAIRˆ²-compliant scientific dataset using the Croissant schema and the `mlcroissant` Python library.

- **Data sourcing:** Used machine-readable metadata for data exploration.
- **Entity referencing:** All fields, record sets, and columns are referenced via their `@id`.
- **Data loading & EDA:** Showed dynamic DataFrame loading, metadata access, statistical processing, and basic visualization.

Refer to the dataset's Croissant schema documentation for detailed understanding of its semantic structure and to extend this notebook for statistical analysis, modeling, or downstream applications.